In [53]:
import os
import glob
import json
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.io import MemoryFile
from shapely.geometry import box
import numpy as np 
import geopandas as gpd


In [68]:
class GEO_PreProcessing:

    def __init__(self):
        self.global_counter = 0  # Track counter across multiple calls

    def open(self, path, no_channels=[1,2,3], transpose=True): 
        """Open an image and optionally select channels""" 
        src = rasterio.open(path) 
        img = src.read(no_channels) 

        if transpose: 
            img = img.transpose(1, 2, 0)
            return src, img
        else:
             return src,img

    
    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3]):
        
        '''
        Find the overlapping area an save both raster within intersection extent

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        
        '''
        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:

            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def align_to_ref(self, ref_src, src_src):
        """
        Align src raster to ref raster grid (CRS, resolution, extent).
        Returns aligned array and its updated transform.
        """
        # Create output array matching reference dimensions
        aligned_data = np.empty(
            (src_src.count, ref_src.height, ref_src.width), 
            dtype=src_src.dtypes[0]
        )

        # Reproject each band
        for i in range(src_src.count):
            reproject(
                source=rasterio.band(src_src, i + 1),
                destination=aligned_data[i],
                src_transform=src_src.transform,
                src_crs=src_src.crs,
                dst_transform=ref_src.transform,
                dst_crs=ref_src.crs,
                resampling=Resampling.bilinear
            )

        print("[ALIGN DONE] Aligned image to reference grid (in-memory).")
        return aligned_data, ref_src.transform

    def makeChips(self, in_pre, in_post, out_pre, out_post, no_channels=[1, 2, 3], chip_size=224, stride=224, file_prefix="", reset_counter=False):
        """
        Creates chips from pre and post images, automatically aligning or cropping if needed.
        Returns chips that are spatially aligned.
        
        Args:
            file_prefix: Optional prefix for output filenames to avoid conflicts
            reset_counter: If True, reset global counter to 0
        """
        os.makedirs(out_pre, exist_ok=True)
        os.makedirs(out_post, exist_ok=True)
        
        # Reset counter if needed
        if reset_counter:
            self.global_counter = 0
        
        counter = self.global_counter

        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:

            # Check if alignment is needed
            needs_alignment = (
                pre_src.crs != post_src.crs or
                pre_src.transform != post_src.transform or
                pre_src.height != post_src.height or
                pre_src.width != post_src.width
            )

            if needs_alignment:
                print("[MISMATCH] Images need alignment. Aligning post to pre...")
                
                aligned_post_data, aligned_transform = self.align_to_ref(pre_src, post_src)
                
                # Create in-memory raster for aligned post
                memfile = MemoryFile()
                aligned_post_src = memfile.open(
                    driver="GTiff",
                    height=pre_src.height,
                    width=pre_src.width,
                    count=post_src.count,
                    dtype=post_src.dtypes[0],
                    crs=pre_src.crs,
                    transform=pre_src.transform
                )
                aligned_post_src.write(aligned_post_data)
                
                print(f"[INFO] Starting chipping from aligned images ({pre_src.height}x{pre_src.width})...")
                
                for row in range(0, pre_src.height - chip_size + 1, stride):
                    for col in range(0, pre_src.width - chip_size + 1, stride):
                        window = Window(col, row, chip_size, chip_size)
                        
                        pre_chip = pre_src.read(no_channels, window=window)
                        post_chip = aligned_post_src.read(no_channels, window=window)
                        
                        chip_transform = rasterio.windows.transform(window, pre_src.transform)
                        
                        chip_meta = {
                            "driver": "GTiff",
                            "height": chip_size,
                            "width": chip_size,
                            "count": len(no_channels),
                            "dtype": pre_src.dtypes[0],
                            "crs": pre_src.crs,
                            "transform": chip_transform
                        }
                        
                        pre_filename = os.path.join(out_pre, f"{file_prefix}pre_{counter}.tif")
                        post_filename = os.path.join(out_post, f"{file_prefix}post_{counter}.tif")
                        
                        with rasterio.open(pre_filename, "w", **chip_meta) as pre_dest:
                            pre_dest.write(pre_chip)
                        
                        with rasterio.open(post_filename, "w", **chip_meta) as post_dest:
                            post_dest.write(post_chip)
                        
                        counter += 1
                
                aligned_post_src.close()
                memfile.close()
                
            else:
                # Images already aligned
                print("[INFO] Images are already aligned. Proceeding with chipping...")
                
                for row in range(0, pre_src.height - chip_size + 1, stride):
                    for col in range(0, pre_src.width - chip_size + 1, stride):
                        window = Window(col, row, chip_size, chip_size)
                        
                    
                        pre_chip = pre_src.read(no_channels, window=window)
                        post_chip = post_src.read(no_channels, window=window)
                    
                        chip_transform = rasterio.windows.transform(window, pre_src.transform)
                        
                        chip_meta = {
                            "driver": "GTiff",
                            "height": chip_size,
                            "width": chip_size,
                            "count": len(no_channels),
                            "dtype": pre_src.dtypes[0],
                            "crs": pre_src.crs,
                            "transform": chip_transform
                        }
                        
                        pre_filename = os.path.join(out_pre, f"{file_prefix}pre_{counter}.tif")
                        post_filename = os.path.join(out_post, f"{file_prefix}post_{counter}.tif")
                        
                        with rasterio.open(pre_filename, "w", **chip_meta) as pre_dest:
                            pre_dest.write(pre_chip)
                        
                        with rasterio.open(post_filename, "w", **chip_meta) as post_dest:
                            post_dest.write(post_chip)
                        
                        counter += 1

            print(f"[DONE] Created {counter - self.global_counter} chips from this image pair.")
            print(f"[TOTAL] {counter} total chips created so far.")
            
            self.global_counter = counter

    def Mean_Std(self, IMAGE_FOLDER_PATH, no_channels):
        '''
        Computing the Mean and Std for the whole dataset using Welford's online algorithm

        Args:
            IMAGE_FOLDER_PATH(str): Provide the directory consist of images (SAMPLE: ./ROOT_DATA_FOLDER/PRE_FOLDER)
            no_channels(list): List of channel indices to read
        '''

        try:
            # Initialize variables BEFORE the loop
            n_pixels = 0
            mean = None
            M2 = None

            for img_name in os.listdir(IMAGE_FOLDER_PATH):
                _,img = self.open(os.path.join(IMAGE_FOLDER_PATH, img_name), no_channels, transpose=False)
                
                # Reshape: (C, H, W) -> (H*W, C)
                pixels = img.reshape(img.shape[0], -1).T 

                # Initialize mean and M2 with correct shape on first image
                if mean is None:
                    mean = np.zeros(pixels.shape[1])
                    M2 = np.zeros(pixels.shape[1])

                n = pixels.shape[0]
                new_mean = np.mean(pixels, axis=0)
                new_M2 = np.var(pixels, axis=0) * n

                delta = new_mean - mean
                total_n = n_pixels + n
                mean += delta * n / total_n
                M2 += new_M2 + delta**2 * n_pixels * n / total_n
                n_pixels = total_n  # Update n_pixels for next iteration

            std = np.sqrt(M2 / n_pixels)
            print(f'[MEAN]----{mean}')
            print(f'[STD]----{std}')
            
        
        except Exception as e:
            print(f'Error: {e}')
            return None, None

In [69]:
process = GEO_PreProcessing()

In [70]:
prepath = r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippre'
                 
postpath = r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippost'

In [71]:
process.Mean_Std(postpath,no_channels=[1,2,3])

Error: 'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippost\post_1007.tif.aux.xml' not recognized as being in a supported file format.


(None, None)

In [ ]:
# prelist = [
#     os.path.join(folderpath, 'pre', img)
#     for img in os.listdir(os.path.join(folderpath, 'pre'))
#     if img.endswith('.tif')
# ]

# postlist = [
#     os.path.join(folderpath, 'post', img)
#     for img in os.listdir(os.path.join(folderpath, 'post'))
#     if img.endswith('.tif')
# ]

In [2]:
# for i in range(0,len(prelist)):
#     process.makeChips(prelist[i],postlist[i],r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippre', r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippost')